In [7]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [10]:
# Membaca file langsung dari path Google Drive
file_path = r'D:\Coding Project\Perancangan Big data\train.parquet'
df = pl.scan_parquet(file_path)

# Menampilkan 5 data teratas
df.head(5).collect()

ip,app,device,os,channel,click_time,attributed_time,is_attributed
i64,i64,i64,i64,i64,str,str,i64
83230,3,1,13,379,"""2017-11-06 14:32:21""",null,0
17357,3,1,19,379,"""2017-11-06 14:33:34""",null,0
35810,3,1,13,379,"""2017-11-06 14:34:12""",null,0
45745,14,1,13,478,"""2017-11-06 14:34:52""",null,0
161007,3,1,13,379,"""2017-11-06 14:35:08""",null,0


In [11]:
# Menghitung jumlah nilai unik (cardinality) untuk setiap kolom kategori
# Ini membantu kita paham seberapa beragam datanya
unique_counts = df.select([
    pl.col("app").n_unique(),
    pl.col("device").n_unique(),
    pl.col("os").n_unique(),
    pl.col("channel").n_unique(),
    pl.col("ip").n_unique()
]).collect()

print("Jumlah Nilai Unik per Kolom:")
display(unique_counts)

: 

In [ ]:
# Cek distribusi target variable (is_attributed)
# Penting untuk tau apakah datanya seimbang atau tidak (imbalanced)
target_dist = df.group_by("is_attributed").agg(pl.count()).collect()

print("Distribusi Target (is_attributed):")
display(target_dist)

/tmp/ipykernel_11615/1886434726.py:3: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  target_dist = df.group_by("is_attributed").agg(pl.count()).collect()


Distribusi Target (is_attributed):


is_attributed,count
i64,u32
1,456846
0,184447044


### 1. Visualisasi Distribusi Target
Karena perbedaan jumlah 0 dan 1 sangat ekstrem, kita gunakan skala logaritmik agar perbedaannya terlihat jelas.

In [1]:
# Konversi ke pandas untuk plotting
target_pd = target_dist.to_pandas()

plt.figure(figsize=(8, 5))
sns.barplot(x='is_attributed', y='count', data=target_pd)
plt.yscale('log')
plt.title('Distribusi Target (is_attributed) - Skala Log')
plt.ylabel('Count (Log Scale)')
plt.show()

NameError: name 'target_dist' is not defined

### 2. Analisis Waktu (Time Series)
Kita akan mengekstrak jam dari `click_time` untuk melihat jam-jam sibuk klik.

In [ ]:
# Ekstrak jam dari click_time
# Karena data besar, kita ambil sample atau agregasi langsung di Polars
hourly_counts = df.with_columns(
    pl.col("click_time").str.to_datetime().dt.hour().alias("click_hour")
).group_by("click_hour").agg(pl.len()).sort("click_hour").collect()

# Plot tren per jam
plt.figure(figsize=(12, 6))
sns.lineplot(x=hourly_counts['click_hour'], y=hourly_counts['len'], marker='o')
plt.title('Jumlah Klik Berdasarkan Jam dalam Sehari')
plt.xlabel('Jam (0-23)')
plt.ylabel('Jumlah Klik')
plt.grid(True)
plt.show()

### 3. Top 10 Apps & Channels
Melihat aplikasi dan saluran mana yang paling mendominasi trafik.

In [ ]:
top_apps = df.group_by("app").agg(pl.len()).sort("len", descending=True).head(10).collect()
top_channels = df.group_by("channel").agg(pl.len()).sort("len", descending=True).head(10).collect()

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x='app', y='len', data=top_apps.to_pandas(), ax=ax[0], palette='viridis')
ax[0].set_title('Top 10 Apps by Traffic')

sns.barplot(x='channel', y='len', data=top_channels.to_pandas(), ax=ax[1], palette='magma')
ax[1].set_title('Top 10 Channels by Traffic')

plt.tight_layout()
plt.show()

### 4. Feature Engineering
Kita akan membuat fitur baru: mengekstrak jam dari waktu klik dan menghitung frekuensi klik per IP.

In [ ]:
import polars as pl

# 1. Pastikan data dimuat
file_path = '/content/drive/MyDrive/presentasi big data/talking data adtracking/hasil extract/train.parquet'
df = pl.scan_parquet(file_path)

# 2. Feature Engineering
df_features = df.with_columns([
    pl.col("click_time").str.to_datetime(),
]).with_columns([
    pl.col("click_time").dt.hour().alias("hour"),
    pl.col("click_time").dt.day().alias("day"),
])

# Menghitung jumlah klik per IP
ip_counts = df_features.group_by("ip").agg(pl.len().alias("ip_click_count"))

# Join kembali ke dataframe utama
df_final = df_features.join(ip_counts, on="ip", how="left")

# Menampilkan hasil
print("Feature Engineering Selesai:")
display(df_final.select(["ip", "app", "device", "os", "channel", "hour", "ip_click_count", "is_attributed"]).head(5).collect())

In [ ]:
import polars as pl

# 1. Muat data (untuk memastikan variabel 'df' tersedia di sesi ini)
file_path = '/content/drive/MyDrive/presentasi big data/talking data adtracking/hasil extract/train.parquet'
df = pl.scan_parquet(file_path)

# 2. Feature Engineering menggunakan Polars LazyFrame
df_features = df.with_columns([
    pl.col("click_time").str.to_datetime(),
]).with_columns([
    pl.col("click_time").dt.hour().alias("hour"),
    pl.col("click_time").dt.day().alias("day"),
])

# Menghitung jumlah klik per IP (fitur penting untuk deteksi fraud)
ip_counts = df_features.group_by("ip").agg(pl.len().alias("ip_click_count"))

# Join kembali ke dataframe utama
df_final = df_features.join(ip_counts, on="ip", how="left")

# Menampilkan hasil transformasi
print("Feature Engineering Selesai:")
display(df_final.select(["ip", "app", "device", "os", "channel", "hour", "ip_click_count", "is_attributed"]).head(5).collect())

In [ ]:
import polars as pl

file_path = r'D:\Coding Project\Perancangan Big data\train.parquet'
df = pl.scan_parquet(file_path)

# Hitung conversion rate
stats = df.select([
    pl.len().alias("total_clicks"),
    (pl.col("is_attributed") == 1).sum().alias("conversions")
]).with_columns([
    (pl.col("conversions") / pl.col("total_clicks") * 100).alias("conversion_rate_%")
]).collect()

print("=" * 50)
print("CONVERSION RATE ANALYSIS")
print("=" * 50)
print(f"Total Clicks: {stats['total_clicks'][0]:,}")
print(f"Conversions: {stats['conversions'][0]:,}")
print(f"Conversion Rate: {stats['conversion_rate_%'][0]:.4f}%")
print("=" * 50)